# Hardware-Software Co-verification
In this notebook, we compare the output of our hardware Rule 30 PRNG with the pure-Python golden model.
Because the hardware core runs freely at 100MHz, an AXI read from Python will capture a state many generations ahead. To verify exact correctness, we will seed the hardware, take a few samples, and prove mathematically that these samples exist in the software-computed Rule 30 trace for that seed.

In [ ]:
pip install golden_model rule30_driver pynq

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement golden_model (from versions: none)
ERROR: No matching distribution found for golden_model


In [ ]:
import sys
sys.path.insert(0, '../hardware/ip_repo/rule30_axi_v1_0/tb')
sys.path.insert(0, '../software/driver')
from golden_model import run as golden_run
from rule30_driver import Rule30PRNG
from pynq import Overlay

import warnings
warnings.filterwarnings('ignore')

# Load Hardware overlay
print('Loading overlay...')
prng_hw = Rule30PRNG(bitfile='../hardware/rule30_design.bit')
print('Overlay loaded.')

In [ ]:
TEST_SEED = 0x12345678
SEARCH_DEPTH = 1000000 # Generate 1M software states to search

print('1. Seeding hardware...')
prng_hw.seed(TEST_SEED)
hw_samples = prng_hw.stream(10)
print(f'Hardware samples grabbed: {[hex(s) for s in hw_samples]}')

print('\n2. Generating Software Trace (Golden Model)...')
sw_trace_gen = golden_run(seed=TEST_SEED, generations=SEARCH_DEPTH)

sw_trace = []
for i, state in enumerate(sw_trace_gen):
    sw_trace.append(state)

print(f'Generated {len(sw_trace)} states in software.')

print('\n3. Comparing Hardware output against Software Trace...')
all_matched = True
for sample in hw_samples:
    if sample in sw_trace:
        idx = sw_trace.index(sample)
        print(f'MATCH: Hardware sample {hex(sample)} found in software trace at generation {idx}')
    else:
        print(f'FAIL: Hardware sample {hex(sample)} NOT FOUND in first {SEARCH_DEPTH} software generations!')
        all_matched = False

if all_matched:
    print('\nSUCCESS: Hardware perfectly matches the Software Golden Model!')
else:
    print('\nERROR: Mismatch detected between hardware and software.')